# 01 · NumPy for AI engineers

After this notebook you can read and write the array code underneath embeddings, attention, sampling and evaluation: dtypes and memory, shapes, broadcasting, batched matmul, cosine similarity, top-k, a stable softmax, masks, seeds and percentiles.

**Time:** ~40 min · **Runs:** offline on CPU in < 30 s · **Needs:** [lists & tuples](../01_python_basics/03_lists_and_tuples.ipynb) and [00 · How to use these notebooks](../00_start_here/00_how_to_use_these_notebooks.ipynb)

## Why this matters in interviews

- Every GenAI system is arrays underneath. An embedding index is an `(N, D)` matrix, attention is batched matmul plus softmax, sampling is a weighted random choice, and an eval report is percentiles. "Implement cosine similarity / softmax / top-k in NumPy" is a standard whiteboard ask.
- Shape bugs are the most common bugs in ML code. Fluency with `axis`, broadcasting and batch shapes is how you avoid them, and how you debug them live.
- dtype × shape arithmetic answers the "will it fit in memory?" questions for embedding indexes and models.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `ve01` | What is cosine similarity, and what are the alternatives? |
| `ve02` | When would you use dot product instead of cosine? |
| `ve23` | What is normalisation and when must you do it? |
| `fo22` | What are logits, and what is the softmax doing? |
| `fo14` | What is temperature actually doing, mathematically? |
| `fo07` | Why does the same prompt give a different answer each time? |
| `pd41` | What is the difference between p50, p95 and p99, and which do you optimise? |
| `tr29` | How do you compute the memory a model needs to serve? |

In [ ]:
import tempfile
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(0)            # one seeded generator for the whole notebook
np.set_printoptions(precision=3, suppress=True)

BLUE, ORANGE, AQUA, GRAY, INK = "#2a78d6", "#eb6834", "#1baf7a", "#b5b3ab", "#52514e"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "axes.axisbelow": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.8})
print("numpy", np.__version__)

## 1. The ndarray: shape, dtype, memory

**In plain English:** an ndarray is one block of memory holding numbers of **one type**, plus a *shape* that says how to read it as a grid. Memory is simply `number of elements × bytes per element`. That one formula sizes embedding indexes, KV caches and model weights.

In [ ]:
E = rng.normal(size=(4, 768)).astype(np.float32)          # 4 embeddings of dimension 768
print(f"shape {E.shape} | ndim {E.ndim} | dtype {E.dtype} | itemsize {E.itemsize} B | nbytes {E.nbytes:,} B")
assert E.nbytes == E.size * E.itemsize == 4 * 768 * 4

print("\n1M embeddings x 768 dims:")
for dt in [np.float64, np.float32, np.float16, np.int8]:
    gb = 1_000_000 * 768 * np.dtype(dt).itemsize / 1e9
    print(f"  {np.dtype(dt).name:8} {np.dtype(dt).itemsize} B/value -> {gb:5.2f} GB")

print("\ndefault dtypes:", np.array([1, 2]).dtype, "|", np.array([1.0]).dtype, "(PyTorch defaults to float32)")

Lower precision has sharp edges. **float16** tops out at 65,504 and cannot represent every integer above 2,048. **int8** silently *wraps around* on overflow. That is why quantization code clips values before casting. (`bfloat16` is not a built-in NumPy dtype; PyTorch has it.)

In [ ]:
with np.errstate(over="ignore"):
    too_big = np.float16(70_000)
print("float16(70000)          :", too_big)                        # overflow -> inf
print("float16(2048) + 1       :", float(np.float16(2048) + np.float16(1)))  # the +1 is lost
print("epsilon fp16 / fp32     :", np.finfo(np.float16).eps, "/", np.finfo(np.float32).eps)
wrapped = np.array([127], dtype=np.int8) + 1
print("int8 array [127] + 1    :", wrapped, "(wraps to -128, no error)")
safe = np.clip(np.array([130.4, -200.0, 5.6]), -127, 127).round().astype(np.int8)
print("clip, round, then cast  :", safe)

assert np.isinf(too_big) and np.float16(2048) + np.float16(1) == 2048
assert wrapped[0] == -128 and safe.tolist() == [127, -127, 6]

## 2. Shapes: reshape, transpose, expand_dims, squeeze

The shape manipulation you will write most often as an AI engineer is **splitting the model dimension into attention heads**: `(B, T, D)` becomes `(B, T, H, D/H)` and then `(B, H, T, D/H)`. `reshape` regroups the elements, and `transpose` reorders the axes. Both are usually *views*: new strides over the same memory, with nothing copied.

In [ ]:
B, T, D, H = 2, 5, 12, 3                       # batch, tokens, model dim, heads
x = np.arange(B * T * D, dtype=np.float32).reshape(B, T, D)
heads = x.reshape(B, T, H, D // H).transpose(0, 2, 1, 3)       # (B, H, T, Dh)
merged = heads.transpose(0, 2, 1, 3).reshape(B, T, D)          # and back again
print("split into heads :", x.shape, "->", heads.shape)
print("merged back      :", merged.shape, "identical:", np.array_equal(merged, x))
assert np.array_equal(merged, x)

v = x[0, 0]                                    # one token vector, shape (12,)
print("add an axis      :", v.shape, "->", v[None, :].shape, "=", np.expand_dims(v, 0).shape)
print("drop a size-1 axis:", x[:, :1].shape, "->", x[:, :1].squeeze(1).shape)
print("-1 is inferred   :", x.reshape(B, -1).shape)
print("views, not copies:", np.shares_memory(x, x.reshape(-1)), np.shares_memory(x, heads),
      "| transposed is C-contiguous:", heads.flags["C_CONTIGUOUS"])

## 3. Indexing: slices, boolean masks, fancy indexing

**Fancy indexing with an integer array is an embedding lookup.** `table[token_ids]` pulls one row per id, and PyTorch's `nn.Embedding` does exactly this. Boolean masks filter. Slices take ranges.

In [ ]:
vocab, d = 10, 4
emb_table = rng.normal(size=(vocab, d)).astype(np.float32)
token_ids = np.array([[3, 1, 4, 1], [5, 9, 2, 6]])                  # a batch: 2 sequences x 4 tokens
vectors = emb_table[token_ids]
print("lookup:", token_ids.shape, "ids ->", vectors.shape, "vectors")
assert vectors.shape == (2, 4, 4) and np.array_equal(vectors[0, 1], emb_table[1])

scores = rng.random(8)
print("scores            :", scores)
print("slice [2:5]       :", scores[2:5])
print("mask scores > 0.5 :", scores[scores > 0.5])
print("where are they?   :", np.flatnonzero(scores > 0.5))
print("rows 0 and 2, col 1:", emb_table[[0, 2], 1])
print("pairs (0,1),(2,3) :", emb_table[[0, 2], [1, 3]])            # element-wise pairs, not a block

## 4. Broadcasting

**In plain English:** when two arrays have different shapes, NumPy lines the shapes up **from the right** and stretches any dimension of size 1 to match the other array. It does this virtually, without copying memory. Two dimensions are compatible if they are equal or one of them is 1. A missing leading dimension counts as 1.

```
(100, 3) - (3,)        ->  (100, 3)      subtract the per-feature mean
(4, 1, 8) - (1, 6, 8)  ->  (4, 6, 8)     every query minus every document
(3, 4) + (3,)          ->  error         trailing dims 4 vs 3 -> write (3, 1)
```

In [ ]:
def draw_grid(ax, arr, x0, title, virtual=None):
    """Draw a 2-D array as boxes; `virtual` marks cells that broadcasting stretches (no memory)."""
    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            v = virtual is not None and virtual[i, j]
            ax.add_patch(plt.Rectangle((x0 + j, -i), 0.92, 0.92, fc="#f4f4f2" if v else "#e7f0ff",
                                       ec="#c3c2b7" if v else "#3b5bdb", lw=1))
            ax.text(x0 + j + 0.46, -i + 0.46, f"{arr[i, j]:g}", ha="center", va="center",
                    fontsize=10, color="#898781" if v else INK)
    ax.text(x0 + arr.shape[1] / 2, 1.35, title, ha="center", fontsize=10)

col, row = np.array([[0], [10], [20]]), np.array([[0, 1, 2, 3]])
fig, ax = plt.subplots(figsize=(11, 2.9))
draw_grid(ax, np.repeat(col, 4, axis=1), 0, "a: shape (3, 1)", virtual=np.arange(4)[None, :].repeat(3, 0) > 0)
ax.text(4.45, -0.55, "+", fontsize=20, ha="center", va="center")
draw_grid(ax, np.repeat(row, 3, axis=0), 5, "b: shape (1, 4)", virtual=np.arange(3)[:, None].repeat(4, 1) > 0)
ax.text(9.45, -0.55, "=", fontsize=20, ha="center", va="center")
draw_grid(ax, col + row, 10, "a + b: shape (3, 4)")
ax.set_xlim(-0.3, 14.3); ax.set_ylim(-2.3, 1.9); ax.axis("off")
ax.set_title("Broadcasting: gray cells are virtual copies, stretched along the size-1 axis", pad=12)
plt.show()

In [ ]:
X = rng.normal(5, 2, size=(100, 3))                     # 100 rows, 3 features
centered = X - X.mean(axis=0)                           # (100, 3) - (3,)
assert np.allclose(centered.mean(axis=0), 0)

queries, docs = rng.normal(size=(4, 8)), rng.normal(size=(6, 8))
diff = queries[:, None, :] - docs[None, :, :]           # (4, 1, 8) - (1, 6, 8) -> (4, 6, 8)
dist = np.sqrt((diff ** 2).sum(axis=-1))                # all 24 query-document distances at once
print("pairwise diff:", diff.shape, "-> distances:", dist.shape)
assert np.isclose(dist[2, 5], np.linalg.norm(queries[2] - docs[5]))

try:
    np.ones((3, 4)) + np.ones(3)                        # trailing dimensions 4 vs 3
    raise AssertionError("should not broadcast")
except ValueError as e:
    print("ValueError:", e)
print("fixed with [:, None]:", (np.ones((3, 4)) + np.ones(3)[:, None]).shape)

## 5. Vectorization vs Python loops

**In plain English:** a Python loop runs the interpreter once per number. A NumPy call runs one compiled loop in C, using SIMD and BLAS, over the whole array. The gap is typically 10 to 1000×, and it grows with the data. Here is the same cosine-similarity search both ways:

In [ ]:
def cosine_loop(q, docs):                  # pure Python, what you'd write without NumPy
    qn = sum(v * v for v in q) ** 0.5
    out = []
    for doc in docs:
        dot = sum(a * b for a, b in zip(q, doc))
        out.append(dot / (qn * sum(v * v for v in doc) ** 0.5))
    return out

def cosine_numpy(q, docs):
    return docs @ q / (np.linalg.norm(docs, axis=1) * np.linalg.norm(q))

def best_time(fn, *args, repeat=3):
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter(); fn(*args); times.append(time.perf_counter() - t0)
    return min(times)

sizes, t_loop, t_np = [100, 300, 1_000, 3_000], [], []
for n in sizes:
    D_ = rng.normal(size=(n, 128)); q = rng.normal(size=128)
    assert np.allclose(cosine_loop(q.tolist(), D_.tolist()), cosine_numpy(q, D_))
    t_loop.append(best_time(cosine_loop, q.tolist(), D_.tolist(), repeat=1))
    t_np.append(best_time(cosine_numpy, q, D_))
print(f"3,000 docs x 128 dims: loop {t_loop[-1] * 1e3:.1f} ms vs NumPy {t_np[-1] * 1e3:.3f} ms "
      f"({t_loop[-1] / t_np[-1]:.0f}x faster)")
assert t_np[-1] < t_loop[-1]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(sizes, np.array(t_loop) * 1e3, marker="o", color=GRAY, label="Python loop")
ax.plot(sizes, np.array(t_np) * 1e3, marker="o", color=BLUE, label="NumPy (vectorised)")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("number of documents (128-dim)"); ax.set_ylabel("time (ms, log scale)")
ax.set_title("Cosine search: the loop grows with the data, NumPy barely moves")
ax.legend(); plt.show()

## 6. Axis semantics

**The rule:** the axis you name is the one that **disappears**. On a batch of logits shaped `(batch, classes)`, `argmax(axis=-1)` gives one prediction per example, and `mean(axis=0)` gives one value per class. `keepdims=True` keeps the reduced axis as size 1, so the result broadcasts straight back.

In [ ]:
logits = rng.normal(size=(4, 5))                 # 4 examples, 5 classes
print("argmax over classes (axis=-1):", logits.argmax(axis=-1), "shape", logits.argmax(axis=-1).shape)
print("mean over the batch  (axis=0):", logits.mean(axis=0).shape)
print("max with keepdims            :", logits.max(axis=1, keepdims=True).shape, "-> broadcasts against (4, 5)")

**Mean pooling with a padding mask** turns token vectors into one sentence embedding (it is what sentence-transformers does). Padding positions must not count. A plain `.mean(axis=1)` averages the padding in:

In [ ]:
tok = rng.normal(size=(2, 4, 3))                  # 2 sequences x 4 positions x 3 dims
mask = np.array([[1, 1, 1, 0],                    # 1 = real token, 0 = padding
                 [1, 1, 0, 0]])
pooled = (tok * mask[:, :, None]).sum(axis=1) / mask.sum(axis=1, keepdims=True)
naive = tok.mean(axis=1)
print("masked mean pooling :", pooled.shape)
print("error of naive mean :", np.abs(pooled - naive).max(axis=1), "(padding leaked in)")
assert np.allclose(pooled[1], tok[1, :2].mean(axis=0))          # only the 2 real tokens count

## 7. Matmul `@`, batch shapes and `einsum`

`@` multiplies the **last two** axes and broadcasts over all the leading ones. That is exactly what attention needs: `(B, H, T, Dh) @ (B, H, Dh, T)` gives `(B, H, T, T)`. `einsum` spells out the same contraction with index letters. It is handy when the transposes get confusing.

| Operation | Meaning | Shapes |
|---|---|---|
| `a * b` | element-wise (broadcast) | `(3,4) * (4,)` → `(3,4)` |
| `a @ b` | matrix product over the last two axes, batched over the rest | `(B,T,D) @ (D,E)` → `(B,T,E)` |
| `np.dot(a, b)` on 3-D+ | a *different* contraction, not batched | `(2,3,4)·(2,4,5)` → `(2,3,2,5)` |
| `np.einsum("ij,jk->ik")` | matmul, written with indices | any |

In [ ]:
B, H, T, Dh = 2, 3, 4, 8
Q, K = rng.normal(size=(B, H, T, Dh)), rng.normal(size=(B, H, T, Dh))
scores = Q @ K.transpose(0, 1, 3, 2) / np.sqrt(Dh)                 # (B,H,T,Dh) @ (B,H,Dh,T)
scores_einsum = np.einsum("bhtd,bhsd->bhts", Q, K) / np.sqrt(Dh)
print("attention scores:", scores.shape)
assert np.allclose(scores, scores_einsum)

W = rng.normal(size=(Dh, 16))
print("project every head's tokens at once:", (Q @ W).shape)

A, Bm = rng.normal(size=(2, 3, 4)), rng.normal(size=(2, 4, 5))
print("A @ B:", (A @ Bm).shape, "| np.dot(A, B):", np.dot(A, Bm).shape, "<- not a batched matmul")
assert (A @ Bm).shape == (2, 3, 5) and np.dot(A, Bm).shape == (2, 3, 2, 5)

u, w = rng.normal(size=5), rng.normal(size=5)
M = rng.normal(size=(3, 5))
assert np.isclose(np.einsum("i,i->", u, w), u @ w)                  # dot product
assert np.allclose(np.einsum("bi,bi->b", M, M), (M * M).sum(1))     # row-wise dot products
assert np.allclose(np.einsum("ij->ji", M), M.T)                     # transpose

## 8. Norms, L2 normalisation and cosine similarity

**In plain English:** cosine similarity measures the *angle* between two vectors and ignores their length. Divide every vector by its length (L2 normalisation), and one matmul gives the similarity of every pair. After normalisation, **dot product = cosine**, which is why vector databases with normalised embeddings use inner product. Without normalisation, dot product also rewards long vectors. Sometimes that is intended, for models trained with dot product, but usually it is a bug.

In [ ]:
topics = ["refund", "password", "shipping"]
centers = rng.normal(size=(3, 64))
emb = np.vstack([c + 0.7 * rng.normal(size=(4, 64)) for c in centers])   # 12 fake sentence embeddings
labels = [f"{t}-{i}" for t in topics for i in range(1, 5)]

unit = emb / np.maximum(np.linalg.norm(emb, axis=1, keepdims=True), 1e-12)   # eps guards zero vectors
S = unit @ unit.T                                                            # (12, 12) cosine matrix
assert np.allclose(np.linalg.norm(unit, axis=1), 1) and np.allclose(np.diag(S), 1)

a, b = unit[0], unit[1]
cos = lambda x, y: x @ y / (np.linalg.norm(x) * np.linalg.norm(y))
print(f"dot(a, b)   = {a @ b:.3f} | cosine = {cos(a, b):.3f}   (equal: both have length 1)")
print(f"dot(a, 3*b) = {a @ (3 * b):.3f} | cosine = {cos(a, 3 * b):.3f}   (dot grew, the angle didn't)")
assert np.isclose(a @ b, cos(a, b)) and np.isclose(cos(a, 3 * b), cos(a, b))

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 5))
im = ax.imshow(S, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(12), labels, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(12), labels, fontsize=8)
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.8, label="cosine similarity")
ax.set_title("Cosine similarity of 12 embeddings: same-topic blocks light up")
plt.tight_layout(); plt.show()

within = S[:4, :4][np.triu_indices(4, 1)].mean()
across = S[:4, 4:].mean()
print(f"mean similarity within a topic {within:.2f} vs across topics {across:.2f}")
assert within > across

## 9. Top-k: `argpartition` vs `argsort`

Retrieval ends with "give me the k best scores". `argsort` sorts all n scores, which is O(n log n). `argpartition(-scores, k)` only guarantees that the k largest come first, in some order, and runs in O(n). Then you sort just those k. At a million scores that is a clear win:

In [ ]:
many = rng.random(1_000_000).astype(np.float32)
k = 5
t0 = time.perf_counter(); top_sort = np.argsort(-many)[:k]; t_sort = time.perf_counter() - t0
t0 = time.perf_counter()
part = np.argpartition(-many, k)[:k]                   # the k largest, unordered: O(n)
top_part = part[np.argsort(-many[part])]               # order just those k: O(k log k)
t_part = time.perf_counter() - t0
print(f"argsort: {t_sort * 1e3:.1f} ms | argpartition + sort k: {t_part * 1e3:.1f} ms")
print("top-5 scores:", np.array2string(many[top_part], precision=7))
assert np.array_equal(top_sort, top_part) and t_part < t_sort

top3 = np.argpartition(-S, 3, axis=1)[:, :3]           # works per row too (here: 3 nearest for each embedding)
print("row 0's 3 nearest (unordered):", sorted(labels[i] for i in top3[0]))

## 10. Softmax: numerically stable, log-sum-exp, and temperature

**In plain English:** the model outputs **logits**, one raw score per token in the vocabulary. Softmax turns them into probabilities: exponentiate, then divide by the sum. `exp(1000)` overflows to `inf`, but softmax does not change if you subtract the same number from every logit. So subtract the max first. **Log-sum-exp** is the same trick for log-probabilities, and it is what cross-entropy uses internally.

In [ ]:
def softmax(z, axis=-1, temperature=1.0):
    z = np.asarray(z, dtype=np.float64) / temperature
    z = z - z.max(axis=axis, keepdims=True)            # shift: exp can no longer overflow
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

def softmax_naive(z):
    e = np.exp(z)
    return e / e.sum()

big = np.array([1000.0, 1001.0, 1002.0])
with np.errstate(over="ignore", invalid="ignore"):
    naive_big = softmax_naive(big)
print("naive  softmax([1000, 1001, 1002]):", naive_big, "<- inf / inf")
print("stable softmax([1000, 1001, 1002]):", softmax(big))
assert np.isnan(naive_big).all() and np.allclose(softmax(big), softmax(big - 1000))
assert np.allclose(softmax([1.0, 2.0, 3.0]), softmax_naive(np.array([1.0, 2.0, 3.0])))

logsumexp = lambda z: z.max() + np.log(np.exp(z - z.max()).sum())
log_probs = big - logsumexp(big)                       # log-softmax, never touching exp(1000)
assert np.allclose(np.exp(log_probs), softmax(big))

P = softmax(rng.normal(size=(4, 10)), axis=-1)         # a batch: one distribution per row
print("batch of 4 rows, each sums to:", P.sum(axis=-1))
assert np.allclose(P.sum(axis=-1), 1)

**Temperature** divides the logits before the softmax: `softmax(z / T)`. A low T sharpens the distribution toward the top token, and as T → 0 it becomes greedy decoding. A high T flattens it toward uniform. It never changes the *order* of the tokens, only how peaked the distribution is.

In [ ]:
vocab = np.array(["Paris", "Lyon", "France", "the", "a", "banana"])
next_logits = np.array([3.0, 2.0, 1.0, 0.5, 0.0, -1.0])      # "The capital of France is ..."
temps = [0.5, 1.0, 2.0]

fig, ax = plt.subplots(figsize=(9, 3.4))
width = 0.27
for i, (temp, color) in enumerate(zip(temps, [BLUE, ORANGE, AQUA])):
    ax.bar(np.arange(len(vocab)) + (i - 1) * width, softmax(next_logits, temperature=temp),
           width=width * 0.92, color=color, label=f"T = {temp}")
ax.set_xticks(range(len(vocab)), vocab); ax.set_ylabel("probability")
ax.set_title("Temperature: low T sharpens, high T flattens (the ranking never changes)")
ax.legend(); plt.show()

entropy = lambda p: float(-(p * np.log(p)).sum())
H_by_T = [entropy(softmax(next_logits, temperature=t)) for t in temps]
print("entropy at T = 0.5, 1, 2:", np.round(H_by_T, 3))
assert H_by_T[0] < H_by_T[1] < H_by_T[2]
assert softmax(next_logits, temperature=0.05).max() > 0.99          # T -> 0 approaches greedy argmax
assert all(softmax(next_logits, temperature=t).argmax() == 0 for t in temps)

## 11. One-hot, masks and padding

A batch must be rectangular, so shorter sequences get **padded** and an **attention mask** records which positions are real. The **causal mask** (`np.tril`) lets position t see only positions ≤ t. Masked scores are set to `-inf` so that softmax gives them exactly zero weight.

In [ ]:
ids = np.array([2, 0, 3])
one_hot = np.eye(5)[ids]                       # row i has a single 1 at column ids[i]
print("one-hot of [2, 0, 3]:\n", one_hot)
assert (one_hot.argmax(axis=1) == ids).all() and one_hot.sum() == len(ids)

seqs = [[5, 8, 2], [7, 1], [4, 9, 6, 3]]       # token ids of different lengths
lengths = np.array([len(s) for s in seqs])
PAD = 0
batch = np.full((len(seqs), lengths.max()), PAD)
for i, s in enumerate(seqs):
    batch[i, :len(s)] = s
attn_mask = np.arange(lengths.max())[None, :] < lengths[:, None]    # from lengths, not from `batch != 0`
print("padded batch:\n", batch, "\nattention mask:\n", attn_mask.astype(int))

In [ ]:
T_ = lengths.max()
causal = np.tril(np.ones((T_, T_), dtype=bool))          # query t may attend to keys 0..t
allowed = causal[None, :, :] & attn_mask[:, None, :]     # (batch, query, key): causal AND not padding
raw = rng.normal(size=(len(seqs), T_, T_))
weights = softmax(np.where(allowed, raw, -np.inf), axis=-1)
print("attention weights for sequence 1 (length 2):\n", weights[1].round(2))
assert np.allclose(weights.sum(axis=-1), 1)
assert np.all(weights[~allowed] == 0)                     # future tokens and padding get exactly 0

## 12. Random numbers: `default_rng`, legacy seeding, sampling tokens

Use **`rng = np.random.default_rng(seed)`** and pass `rng` around. The legacy `np.random.seed(0)` sets hidden global state that any other code can consume, which is shown in [quick topic 12](../00_start_here/01_quick_interview_topics.ipynb). **Sampling a token** is `rng.choice(vocab, p=probs)`. This is exactly why the same prompt can give different answers (`fo07`): the model returns the same distribution every time, and the *draw* from it changes. Fix the seed and the draw repeats.

In [ ]:
r = np.random.default_rng(42)
print("integers   :", r.integers(0, 10, size=5))
print("normal     :", r.normal(size=3))
print("permutation:", r.permutation(5))
assert np.array_equal(np.random.default_rng(42).integers(0, 10, size=5), np.random.default_rng(42).integers(0, 10, size=5))

probs = softmax(next_logits)
draws = np.random.default_rng(7).choice(vocab, size=20_000, p=probs)
for word, p_true in zip(vocab, probs):
    print(f"  {word:7} p={p_true:.3f}  sampled {np.mean(draws == word):.3f}")
assert np.allclose([np.mean(draws == w) for w in vocab], probs, atol=0.01)

generate = lambda seed: " ".join(np.random.default_rng(seed).choice(vocab, size=6, p=probs))
print("seed 1:", generate(1)); print("seed 1:", generate(1)); print("seed 2:", generate(2))
assert generate(1) == generate(1) != generate(2)

## 13. Percentiles: p50, p95, p99

`np.percentile(x, [50, 95, 99])` takes values from 0 to 100, and `np.quantile(x, 0.95)` takes fractions. Latency is right-skewed, so the mean sits *above* the median and says nothing about the tail. Report p50/p95/p99. The default interpolates between neighbouring values. `method="inverted_cdf"` gives the classic "nearest rank" value, which is an actual observed latency.

In [ ]:
lat = rng.lognormal(np.log(600), 0.5, size=5_000)          # latencies in ms
p50, p95, p99 = np.percentile(lat, [50, 95, 99])
print(f"mean {lat.mean():.0f} | p50 {p50:.0f} | p95 {p95:.0f} | p99 {p99:.0f} ms")
assert np.isclose(np.quantile(lat, 0.95), p95) and lat.mean() > p50

nearest_rank = np.sort(lat)[int(np.ceil(0.95 * lat.size)) - 1]      # the 4,750th smallest of 5,000
print(f"nearest-rank p95 by hand {nearest_rank:.1f} = np.percentile(..., method='inverted_cdf') "
      f"{np.percentile(lat, 95, method='inverted_cdf'):.1f}")
assert np.isclose(nearest_rank, np.percentile(lat, 95, method="inverted_cdf"))

## 14. Combining and cleaning: `stack`, `concatenate`, `where`, `clip`

`concatenate` joins arrays along an **existing** axis, and `stack` creates a **new** one (a batch). `np.where(cond, a, b)` is a vectorised if/else. `clip` bounds values, for example to keep probabilities away from 0 before a `log`.

In [ ]:
a, b = np.zeros((2, 3)), np.ones((2, 3))
print("concatenate axis 0:", np.concatenate([a, b]).shape, "| axis 1:", np.concatenate([a, b], axis=1).shape)
print("stack      axis 0:", np.stack([a, b]).shape, "| axis -1:", np.stack([a, b], axis=-1).shape)
assert np.concatenate([a, b]).shape == (4, 3) and np.stack([a, b]).shape == (2, 2, 3)

z = np.array([-2.0, -0.5, 0.0, 0.7, 3.0])
relu = np.where(z > 0, z, 0.0)
print("ReLU via where :", relu, "| indices where z > 0:", np.where(z > 0)[0])
print("clip to [-1, 1]:", np.clip(z, -1, 1))
assert np.array_equal(relu, np.maximum(z, 0))

p = np.array([0.0, 0.3, 0.7])
with np.errstate(divide="ignore"):
    print("log(p)            :", np.log(p), "<- -inf poisons a loss")
print("log(clip(p, 1e-12)):", np.log(np.clip(p, 1e-12, 1.0)))

## 15. Views vs copies

A **view** shares memory with the original, so writing to it changes the original. This is the NumPy gotcha interviewers love.

| Operation | Result |
|---|---|
| basic slicing `a[1:4]`, `reshape`, `transpose`, `ravel` (when possible) | **view** |
| boolean or fancy indexing `a[mask]`, `a[[1, 2]]`, `flatten()`, `astype()`, `copy()`, arithmetic | **copy** |
| `b = a` | **the same object**, not even a view |

In [ ]:
base = np.arange(6)
view = base[1:4]
view[0] = 100
print("after writing to a slice:", base, "<- the original changed")
assert base[1] == 100 and np.shares_memory(base, view)

fancy = base[[1, 2]]; fancy[0] = -1
assert base[1] == 100 and not np.shares_memory(base, fancy)        # fancy indexing copied
safe = base[1:4].copy(); safe[0] = 7
assert base[1] == 100                                               # .copy() protects the original
alias = base; alias[0] = 99
assert base[0] == 99                                                # plain assignment copies nothing
print("ravel shares memory:", np.shares_memory(base, base.ravel()),
      "| flatten shares memory:", np.shares_memory(base, base.flatten()))

## 16. Save and load: `.npy`, `.npz`, memory-mapping

`.npy` stores one array with its exact dtype and shape. `.npz` bundles several arrays. `mmap_mode="r"` maps the file instead of reading it, so an embedding matrix bigger than RAM is paged in only as you index it. `np.load` refuses pickled object arrays by default (`allow_pickle=False`). Keep it that way for files you did not create.

In [ ]:
tmp = Path(tempfile.mkdtemp())
emb32 = unit.astype(np.float32)
np.save(tmp / "embeddings.npy", emb32)
loaded = np.load(tmp / "embeddings.npy")
assert np.array_equal(loaded, emb32) and loaded.dtype == np.float32

np.savez_compressed(tmp / "index.npz", embeddings=emb32, ids=np.arange(len(emb32)))
with np.load(tmp / "index.npz") as bundle:
    print("npz contents:", bundle.files, bundle["embeddings"].shape)

mm = np.load(tmp / "embeddings.npy", mmap_mode="r")
print(f"memory-mapped: {type(mm).__name__} {mm.shape}; file is {(tmp / 'embeddings.npy').stat().st_size:,} bytes "
      f"(= {emb32.nbytes:,} data + a small header)")

## Try it yourself

**1.** Cosine top-3 retrieval: build a query that is `unit[5]` (a password sentence) plus a little noise, and return the labels of the 3 most similar embeddings using one matmul and `argpartition`.

In [ ]:
# Solution — try it yourself first, then run this
query = unit[5] + 0.3 * rng.normal(size=unit.shape[1])
query = query / np.linalg.norm(query)
sims = unit @ query                                        # (12,) cosine scores
top = np.argpartition(-sims, 3)[:3]
top = top[np.argsort(-sims[top])]
print([(labels[i], round(float(sims[i]), 3)) for i in top])
assert all(labels[i].startswith("password") for i in top)

**2.** Temperature never changes the argmax. Check this on a batch of 8 random logit rows for T in {0.3, 0.7, 1.5}, and check that every row still sums to 1.

In [ ]:
# Solution — try it yourself first, then run this
L = rng.normal(size=(8, 50))
for temp in [0.3, 0.7, 1.5]:
    P_T = softmax(L, axis=-1, temperature=temp)
    assert np.allclose(P_T.sum(axis=-1), 1) and np.array_equal(P_T.argmax(-1), L.argmax(-1))
print("argmax unchanged and rows sum to 1 for every temperature")

**3.** Make a sentence-embedding function: masked mean pooling, then L2 normalisation. Show that padding a sequence with extra (masked) positions does not change its embedding.

In [ ]:
# Solution — try it yourself first, then run this
def sentence_embedding(tokens, mask):
    pooled = (tokens * mask[..., None]).sum(axis=-2) / mask.sum(axis=-1, keepdims=True)
    return pooled / np.linalg.norm(pooled, axis=-1, keepdims=True)

e1 = sentence_embedding(tok[1, :2], mask[1, :2])                    # the 2 real tokens only
padded_tokens = np.concatenate([tok[1], rng.normal(size=(3, 3))])   # 3 more junk positions ...
padded_mask = np.concatenate([mask[1], np.zeros(3, dtype=int)])     # ... all masked out
e2 = sentence_embedding(padded_tokens, padded_mask)
print("same embedding with and without extra padding:", np.allclose(e1, e2))
assert np.allclose(e1, e2)

**4.** Memory: 10 million embeddings of 1,024 dims. How many GB in float32, float16, and int8 with one float32 scale per vector?

In [ ]:
# Solution — try it yourself first, then run this
n_vec, dim = 10_000_000, 1024
sizes_gb = {"float32": n_vec * dim * 4 / 1e9, "float16": n_vec * dim * 2 / 1e9,
            "int8 + scale": (n_vec * dim * 1 + n_vec * 4) / 1e9}
for name, gb in sizes_gb.items():
    print(f"{name:13} {gb:6.2f} GB")
assert sizes_gb["float32"] == 40.96 and sizes_gb["float16"] == 20.48

## Say it in an interview

- **"Implement cosine similarity for a batch."** L2-normalise the rows (with an epsilon for zero vectors), then `A @ B.T`. That gives every pair in one BLAS call. After normalisation dot product equals cosine, so vector databases store normalised vectors and use inner product (`ve01`, `ve02`, `ve23`).
- **"Why subtract the max in softmax?"** `exp` overflows at around 710 in float64 and around 89 in float32. Softmax is shift-invariant, so subtracting the max changes nothing mathematically and prevents `inf/inf = nan`. Log-sum-exp gives stable log-probabilities, and cross-entropy uses it internally (`fo22`).
- **"What does temperature do?"** It computes `softmax(logits / T)`. T < 1 sharpens and T > 1 flattens. The ranking is unchanged, and T → 0 is greedy decoding (`fo14`). Different answers to the same prompt come from *sampling* from that distribution (`fo07`).
- **"Top-k over a million scores?"** `argpartition` in O(n), then sort the k. Not a full `argsort`.
- **"Explain broadcasting."** Align shapes from the right. Each dimension must be equal or 1. `(N,1,D) - (1,M,D)` gives all pairs. The classic bug is `(N,)` vs `(N,1)`.
- **"How much memory?"** Elements × bytes. 1M × 768 in float32 = 3.07 GB, half that in float16, a quarter in int8. The same arithmetic sizes model weights (`tr29`).
- **Numbers:** fp16 max 65,504 and exact integers only up to 2,048 · float32 eps ≈ 1.2e-7 · `argpartition` O(n) vs `argsort` O(n log n) · p95 means 1 in 20 is slower (`pd41`).
- **Traps:** slices are views and writing to them mutates the original · int8 wraps on overflow silently, so clip before casting · `np.dot` on 3-D arrays is not batched matmul · mean pooling without the mask averages padding in · derive attention masks from lengths, because token id 0 can be real.

## Next

- [02 · pandas essentials](02_pandas_essentials.ipynb): the same vectorised thinking, applied to tables and LLM logs.
- [03 · PyTorch essentials](03_pytorch_essentials.ipynb): tensors are ndarrays plus autograd and devices.
- [04 · matplotlib & quick charts](04_matplotlib_and_quick_charts.ipynb)
- Put it to work: [embeddings & similarity](../07_genai_foundations/02_embeddings_and_similarity.ipynb) · [attention from scratch](../07_genai_foundations/03_attention_from_scratch.ipynb) · [decoding: temperature, top-k, top-p](../07_genai_foundations/05_decoding_temperature_top_k_top_p.ipynb) · [vector search with FAISS & HNSW](../09_rag/03_vector_search_faiss_hnsw.ipynb)
- Theory: [genai_flow course](../../genai_flow/index.html) · [interview bank](../../ai_interview_prep/index.html)